In [ ]:
import os
import re
import math
import random
import torch
import torch.nn as nn
from torch.nn import functional as F
from torch.nn.attention.flex_attention import flex_attention, create_block_mask
# import triton
# import triton.language as tl
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from IPython import display
torch.manual_seed(3047)
torch.set_printoptions(profile="short", sci_mode=False, linewidth=100000)
torch.set_float32_matmul_precision('high')
# this script is configured to run on a RTX 3060 12GB GPU. you'll want to adjust the model sizes and batch sizes for other devices
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
# device = torch.device('cpu')
plt.rcParams['figure.figsize'] = [8, 6]
plt.rcParams['figure.dpi'] = 50
plt.rcParams['axes.grid'] = True
plt.rcParams['xtick.minor.visible'] = True
plt.rcParams['ytick.minor.visible'] = True
# make 'models' folder to save trained models if it doesn't exist
os.makedirs('models', exist_ok=True)
device

device(type='mps')

# Data Prep

In [4]:
from datasets import load_dataset

img_size=224
def transforms_dataset(examples):
    examples["image"] = [image.convert("RGB").resize((img_size,img_size)) for image in examples["image"]]
    return examples

df = load_dataset("Erland/coco_captions_small")
df = df.map(transforms_dataset, batched=True)
df

DatasetDict({
    train: Dataset({
        features: ['image', 'filename', 'cocoid', 'caption'],
        num_rows: 5000
    })
})

In [7]:
from datasets import Dataset, DatasetDict
import base64

if isinstance(df, DatasetDict):
    df = df["train"]
if isinstance(df, Dataset):
    df = df.to_pandas()

df["b64string_images"] = df["image"].apply(lambda x: base64.b64encode(x["bytes"]).decode('utf-8'))
df.head(2)

,image,filename,cocoid,caption,b64string_images
0,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,COCO_val2014_000000522418.jpg,522418,A woman wearing a net on her head cutting a ca...,iVBORw0KGgoAAAANSUhEUgAAAOAAAADgCAIAAACVT/22AA...
1,{'bytes': b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHD...,COCO_val2014_000000522418.jpg,522418,A woman cutting a large white sheet cake.,iVBORw0KGgoAAAANSUhEUgAAAOAAAADgCAIAAACVT/22AA...


In [9]:
text = "".join(df["caption"].tolist())
chars = sorted(list(set(text)))
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
# add for special image pad token
stoi['<pad>']= 65
itos[65] = '<pad>'
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string
vocab_size = len(stoi.keys())

In [14]:
print("First 5 items of stoi (string to integer mapping):")
stoi_items = list(stoi.items())[:5]
for char, idx in stoi_items:
    print(f"'{char}' -> {idx}")

print("\nFirst 5 items of itos (integer to string mapping):")
itos_items = list(itos.items())[:5]
for idx, char in itos_items:
    print(f"{idx} -> '{char}'")

print(f"\nVocabulary size: {vocab_size}")
print("First 5 characters in the vocabulary:")
for i in range(min(5, vocab_size)):
    print(f"{i}: '{itos[i]}'")


First 5 items of stoi (string to integer mapping):
'
' -> 0
' ' -> 1
'"' -> 2
''' -> 3
',' -> 4

First 5 items of itos (integer to string mapping):
0 -> '
'
1 -> ' '
2 -> '"'
3 -> '''
4 -> ','

Vocabulary size: 71
First 5 characters in the vocabulary:
0: '
'
1: ' '
2: '"'
3: '''
4: ','


In [15]:
data = torch.tensor(encode(text), dtype=torch.int64)
data.shape

torch.Size([261994])

In [16]:
data[:100]

tensor([19,  1, 66, 58, 56, 44, 57,  1, 66, 48, 44, 61, 52, 57, 50,  1, 44,  1, 57, 48, 63,  1, 58, 57,  1, 51, 48, 61,  1, 51, 48, 44, 47,  1, 46, 64, 63, 63, 52, 57, 50,  1, 44,  1, 46, 44, 54, 48,  6,  1, 19,  1, 66, 58, 56, 44, 57,  1, 46, 64, 63, 63, 52, 57, 50,  1, 44,  1, 55, 44, 61, 50, 48,  1, 66, 51, 52, 63, 48,  1, 62, 51, 48, 48, 63,  1, 46, 44, 54, 48,  6, 19,  1, 66, 58, 56, 44, 57,  1, 66])

# Training Loop

# Model

In [ ]:
class PatchEmbeddings(nn.Module):
    def __init__(self, img_size=224, patch_size=16, hidden_dim=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.conv = nn.Conv2d(in_channels=3, out_channels=hidden_dim,
                              kernel_size=patch_size, stride=patch_size)

    def forward(self, X):
        X = self.conv(X)
        X = X.flatten(2)  # Flatten the patch dimensions
        X = X.transpose(1, 2)  # [B, num_patches, hidden_dim]
        return X

In [19]:
from transformers import MobileViTFeatureExtractor, MobileViTForImageClassification
from PIL import Image
import requests

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

feature_extractor = MobileViTFeatureExtractor.from_pretrained("apple/mobilevit-small")
vit = MobileViTForImageClassification.from_pretrained("apple/mobilevit-small")
inputs = feature_extractor(images=image, return_tensors="pt")

outputs = vit(**inputs)
logits = outputs.logits
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", vit.config.id2label[predicted_class_idx])


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/patrick.irawan/Desktop/mllm-playground/.venv/lib/python3.10/site-packages/transformers/models/mobilevit/feature_extraction_mobilevit.py:30: FutureWarning: The class MobileViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use MobileViTImageProcessor instead.
  warnings.warn(


Predicted class: tabby, tabby cat


In [24]:
inputs.pixel_values.shape

torch.Size([1, 3, 256, 256])

In [26]:
outputs.logits.shape

torch.Size([1, 1000])